# Conversion of APEBench GSDR -> PDEBench DR/TheWell GSDR
No metadata is needed in this conversion.

Taking 'sims' from APEBench GSDR data:
1) Downsample data from 256x256 to 128x128
2) Tranpose the data order such that it matches PDEBench's DR/TheWell GSDR conventions
3) Data normalization (not shown in this notebook, but already done). Script used are included and named accordingly.
4) Inference using PDEBench DR/TheWell GSDR pipeline

# Conversion of APEBench GSDR -> PDEBench DR

In [6]:
import h5py

gs_alpha_ape = '/Users/sky/Desktop/gs_alpha/gs_alpha.hdf5'

with h5py.File(gs_alpha_ape, 'r') as f:
    print(f.keys())
    print(f['sims'].keys())
    print(f['sims']['sim0'])

<KeysViewHDF5 ['norm_const_max', 'norm_const_mean', 'norm_const_min', 'norm_const_std', 'norm_fields_sca_max', 'norm_fields_sca_mean', 'norm_fields_sca_min', 'norm_fields_sca_std', 'norm_fields_std', 'sims']>
<KeysViewHDF5 ['sim0', 'sim1', 'sim2', 'sim3', 'sim4', 'sim5', 'sim6', 'sim7', 'sim8', 'sim9', 'sim10', 'sim11', 'sim12', 'sim13', 'sim14', 'sim15', 'sim16', 'sim17', 'sim18', 'sim19', 'sim20', 'sim21', 'sim22', 'sim23', 'sim24', 'sim25', 'sim26', 'sim27', 'sim28', 'sim29']>
<HDF5 dataset "sim0": shape (100, 2, 256, 256), type "<f4">


In [11]:
# APE gs_alpha -> PDB DR
import h5py
import numpy as np
from scipy.ndimage import zoom

input_file = '/Users/sky/Desktop/gs_alpha/gs_alpha.hdf5'
output_file = '/Volumes/T7/APEBench/converted/gs_alpha_converted_pdbdr.h5'

target_size = 128

with h5py.File(input_file, 'r') as f_in:
    sims_group = f_in['sims']

    with h5py.File(output_file, 'w') as f_out:
        for i, old_key in enumerate(sims_group.keys()):
            print(old_key)
            data = sims_group[old_key][:]
            print(data.shape)

            zoom_factors = (1, 1, target_size / data.shape[2], target_size / data.shape[3])
            data_downsampled = zoom(data, zoom_factors, order=1)

            data_final = np.transpose(data_downsampled, (0, 2, 3, 1))
            new_group_name = f"{i:04d}"
            group = f_out.create_group(new_group_name)
            group.create_dataset('data', data=data_final, dtype='f4')

            print(f"Processed {old_key} -> {new_group_name}/data | Shape: {data_final.shape}")

print("Done")

sim0
(100, 2, 256, 256)
Processed sim0 -> 0000/data | Shape: (100, 128, 128, 2)
sim1
(100, 2, 256, 256)
Processed sim1 -> 0001/data | Shape: (100, 128, 128, 2)
sim2
(100, 2, 256, 256)
Processed sim2 -> 0002/data | Shape: (100, 128, 128, 2)
sim3
(100, 2, 256, 256)
Processed sim3 -> 0003/data | Shape: (100, 128, 128, 2)
sim4
(100, 2, 256, 256)
Processed sim4 -> 0004/data | Shape: (100, 128, 128, 2)
sim5
(100, 2, 256, 256)
Processed sim5 -> 0005/data | Shape: (100, 128, 128, 2)
sim6
(100, 2, 256, 256)
Processed sim6 -> 0006/data | Shape: (100, 128, 128, 2)
sim7
(100, 2, 256, 256)
Processed sim7 -> 0007/data | Shape: (100, 128, 128, 2)
sim8
(100, 2, 256, 256)
Processed sim8 -> 0008/data | Shape: (100, 128, 128, 2)
sim9
(100, 2, 256, 256)
Processed sim9 -> 0009/data | Shape: (100, 128, 128, 2)
sim10
(100, 2, 256, 256)
Processed sim10 -> 0010/data | Shape: (100, 128, 128, 2)
sim11
(100, 2, 256, 256)
Processed sim11 -> 0011/data | Shape: (100, 128, 128, 2)
sim12
(100, 2, 256, 256)
Processed s

# Sanity checks

In [21]:
import h5py

file = '/Volumes/T7/MORPH_Processed/DR2d_data_pdebench/gs_alpha_converted_pdbdr.h5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['0000'].keys())
    print(f['0000']['data'])

#PDB DR
# <KeysViewHDF5 ['data', 'grid']>
# <HDF5 dataset "data": shape (101, 128, 128, 2), type "<f4">

print()

file2 = '/Volumes/T7/MORPH_Processed/normalized_revin/DR2d_data_pdebench/gs_alpha_converted_pdbdr.h5'
with h5py.File(file2, 'r') as f:
    print(f.keys())
    print(f['0000'].keys())
    print(f['0000']['data'])

<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020', '0021', '0022', '0023', '0024', '0025', '0026', '0027', '0028', '0029']>
<KeysViewHDF5 ['data']>
<HDF5 dataset "data": shape (100, 128, 128, 2), type "<f4">

<KeysViewHDF5 ['0000', '0001', '0002', '0003', '0004', '0005', '0006', '0007', '0008', '0009', '0010', '0011', '0012', '0013', '0014', '0015', '0016', '0017', '0018', '0019', '0020', '0021', '0022', '0023', '0024', '0025', '0026', '0027', '0028', '0029']>
<KeysViewHDF5 ['data']>
<HDF5 dataset "data": shape (100, 128, 128, 2), type "<f4">


# Inference for APEBench GSDR using PDEBench DR's pipeline
As mentioned, I have done data normalization separately, so I will omit the step in this notebook. The script used for the data normalization step is included.

In [22]:
#Converted APEBench GSDR ->PDB DR,
!python MORPH/scripts/infer_MORPH.py \
    --model_choice FM \
    --model_size S \
    --checkpoint morph-S-FM-max_ar1_ep225.pth \
    --test_dataset DR \
    --ar_order 1 \
    --rollout_horizon 10 \
    --batch_size 1 \
    --test_sample 0 \
    --max_ar_order 1

DEBUG: Project Root is set to: /Users/sky/Git/MORPH/MORPH
Number of CUDA devices = 0
No CUDA devices available 

→ Selected Batch size for DR is 1
→ Selected Model MORPH-FM-S for Dataset DR
→ Loading test dataset...
[DR] Importing test data...
→ [DR] Shape of test data: (30, 100, 1, 128, 128, 1, 2)
→[DR] Dataset preparation...
Images(N,T,F,C,D,H,W): torch.Size([2970, 1, 2, 1, 1, 128, 128]), Targets(N,F,C,D,H,W): torch.Size([2970, 2, 1, 1, 128, 128])
→ Length dataloader: 2970
→ NUMBER OF PARAMETERS OF THE ViT (in millions): 32.8
TESTING MODEL:
ViT3DRegression(
  (patch_embedding): HybridPatchEmbedding3D(
    (conv_features): ConvOperator(
      (input_proj): Conv3d(3, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (conv_stack): Sequential(
        (0): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): LeakyReLU(negative_slope=0.2, inplace=True)
      )
    )
    (projection): Linear(in_features=4096, out_features=512, bias=T

# Results
APE gs_alpha -> PDB DR results:

MAE: 0.72567, MSE: 1.72229, RMSE: 1.31236, NRMSE: 0.40347, VRMSE: 0.57027

# Conversion of APEBench GSDR -> TheWell GSDR

There is a need to separate the two channels of sims in APEBench GSDR to ['A','B'] respectively for TheWell GSDR format.

In [24]:
# APE gs_alpha -> TheWell GSDR
import h5py
import numpy as np
from scipy.ndimage import zoom

input_file = '/Users/sky/Desktop/gs_alpha/gs_alpha.hdf5'
output_file = '/Volumes/T7/MORPH_Processed/2dgrayscottdr_thewell/gs_alpha_converted_thewell.h5'

target_size = 128

with h5py.File(input_file, 'r') as f_in:
    sims_group = f_in['sims']
    sim_keys = sorted(sims_group.keys(), key=lambda x: int(x.replace('sim', '')))

    processed_data = []
    for key in sim_keys:
        data = sims_group[key][:]
        zoom_factors = (1, 1, target_size / data.shape[2], target_size / data.shape[3])
        data_downsampled = zoom(data, zoom_factors, order=1)
        processed_data.append(data_downsampled)

    all_sims = np.array(processed_data)
    print(all_sims.shape)

    with h5py.File(output_file, 'w') as f_out:
        t0_fields = f_out.create_group('t0_fields')
        t0_fields.create_dataset('A', data=all_sims[:, :, 0, :, :], dtype='f4')
        t0_fields.create_dataset('B', data=all_sims[:, :, 1, :, :], dtype='f4')

print("Done")

(30, 100, 2, 128, 128)
Done


# Sanity check

In [1]:
import h5py

file = '/Volumes/T7/MORPH_Processed/2dgrayscottdr_thewell/gs_alpha_converted_thewell.h5'

with h5py.File(file, 'r') as f:
    print(f.keys())
    print(f['t0_fields'].keys())
    print(f['t0_fields']['A'])

file2 = '/Volumes/T7/MORPH_Processed/2dgrayscottdr_thewell/thewell/test/gray_scott_reaction_diffusion_bubbles_F_0.098_k_0.057.hdf5'
with h5py.File(file2, 'r') as f:
    print(f.keys())
    print(f['t0_fields'].keys())
    print(f['t0_fields']['A'])

<KeysViewHDF5 ['t0_fields']>
<KeysViewHDF5 ['A', 'B']>
<HDF5 dataset "A": shape (30, 100, 128, 128), type "<f4">
<KeysViewHDF5 ['boundary_conditions', 'dimensions', 'scalars', 't0_fields', 't1_fields', 't2_fields']>
<KeysViewHDF5 ['A', 'B']>
<HDF5 dataset "A": shape (20, 1001, 128, 128), type "<f4">


# Inference for APEBench GSDR using TheWell GSDR's pipeline
As mentioned, I have done data normalization separately, so I will omit the step in this notebook. The script used for the data normalization step is included.

In [4]:
#Converted APEBench GSDR ->PDB DR,
!python MORPH/scripts/infer_MORPH.py \
    --model_choice FM \
    --model_size S \
    --checkpoint morph-S-FM-max_ar1_ep225.pth \
    --test_dataset GSDR2D \
    --ar_order 1 \
    --rollout_horizon 10 \
    --batch_size 1 \
    --test_sample 0 \
    --max_ar_order 1

DEBUG: Project Root is set to: /Users/sky/Git/MORPH/MORPH
Number of CUDA devices = 0
No CUDA devices available 

→ Selected Batch size for GSDR2D is 1
→ Selected Model MORPH-FM-S for Dataset GSDR2D
→ Loading test dataset...
[GSDR] Importing test data...
→ [GSDR2D] Shape of test data: (30, 100, 1, 128, 128, 1, 2)
→[GSDR2D] Dataset preparation...
Images(N,T,F,C,D,H,W): torch.Size([2970, 1, 2, 1, 1, 128, 128]), Targets(N,F,C,D,H,W): torch.Size([2970, 2, 1, 1, 128, 128])
→ Length dataloader: 2970
→ NUMBER OF PARAMETERS OF THE ViT (in millions): 32.8
TESTING MODEL:
ViT3DRegression(
  (patch_embedding): HybridPatchEmbedding3D(
    (conv_features): ConvOperator(
      (input_proj): Conv3d(3, 8, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (conv_stack): Sequential(
        (0): Conv3d(8, 8, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        (1): LeakyReLU(negative_slope=0.2, inplace=True)
      )
    )
    (projection): Linear(in_features=4096, out_fe

# Results
APE gs_alpha -> TheWell GSDR results:

MAE: 0.72567, MSE: 1.72229, RMSE: 1.31236, NRMSE: 0.40347, VRMSE: 0.57027

# Observations/Conclusion

The results for both APEBench GSDR conversions are the same, which indicates that the conversions work as intended.